In [1]:
# to check the time stamps from video and epoch info
from pynwb import NWBHDF5IO
import pandas as pd
import numpy as np

path_to_file = "/home/supraja/Documents/spyglass/NA25/raw/na2520250919.nwb"

# Open NWB file in read/write mode
io = NWBHDF5IO(path_to_file, 'r+')
nwb = io.read()

# Get epoch intervals as a DataFrame
epochs_df = nwb.intervals['epochs'].to_dataframe()
print("Original epoch times:")
print(epochs_df)

# Get video start/stop times
video_ts = nwb.processing['video_files']['video'].time_series
video_df = []
for key in video_ts:
    t = video_ts[key].get_timestamps()
    video_df.append({
        "video": key,
        "start_time": t[0],
        "end_time": t[-1]
    })
video_df = pd.DataFrame(video_df)
print("\nVideo times:")
print(video_df)


Original epoch times:
      start_time     stop_time        tags
id                                        
0   1.758257e+09  1.758258e+09  [01_sleep]
1   1.758258e+09  1.758259e+09    [02_run]
2   1.758259e+09  1.758260e+09  [03_sleep]
3   1.758260e+09  1.758261e+09    [04_run]
4   1.758261e+09  1.758262e+09  [05_sleep]
5   1.758262e+09  1.758263e+09    [06_run]
6   1.758263e+09  1.758263e+09  [07_sleep]

Video times:
                          video    start_time      end_time
0  20250919_na25_01_sleep.1.mp4  1.758257e+09  1.758258e+09
1    20250919_na25_02_run.1.mp4  1.758258e+09  1.758259e+09
2  20250919_na25_03_sleep.1.mp4  1.758259e+09  1.758260e+09
3    20250919_na25_04_run.1.mp4  1.758260e+09  1.758261e+09
4  20250919_na25_05_sleep.1.mp4  1.758261e+09  1.758262e+09
5    20250919_na25_06_run.1.mp4  1.758262e+09  1.758263e+09
6  20250919_na25_07_sleep.1.mp4  1.758263e+09  1.758263e+09


In [ ]:
# mapping the epoch time stamps to align with video time stamps 

import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import h5py
import numpy as np
import pandas as pd
from pynwb import NWBHDF5IO

path = "/media/supraja/NA_2_2025/Supraja/spyglass/Koel/NA3/20250716/raw/na320250716.nwb"

# --- load video times (read-only) ---
io = NWBHDF5IO(path, "r")
nwb = io.read()
video_ts = nwb.processing["video_files"]["video"].time_series
video_rows = []
for key in video_ts:
    t = video_ts[key].get_timestamps()
    video_rows.append({"video": key, "start_time": t[0], "end_time": t[-1], "frames": len(t)})
video_df = pd.DataFrame(video_rows)
print("Video times:\n", video_df)
epochs_df = nwb.intervals["epochs"].to_dataframe()
print("\nEpoch times (before):\n", epochs_df)
io.close()

# --- compute new epoch times from video span ---
video_start = float(video_df.start_time.min())
video_stop = float(video_df.end_time.max())
start_arr = epochs_df["start_time"].to_numpy()
stop_arr = epochs_df["stop_time"].to_numpy()
old_ranges = list(zip(start_arr, stop_arr))
offset = video_start - start_arr[0]
start_new = start_arr + offset
stop_new = stop_arr + offset
stop_new[-1] = video_stop  # force final stop to match last video end
new_ranges = list(zip(start_new, stop_new))

# --- write updates and append session description ---
with h5py.File(path, "r+") as f:
    f["intervals"]["epochs"]["start_time"][...] = start_new
    f["intervals"]["epochs"]["stop_time"][...] = stop_new
    tag = " [INFO] Epoch timestamps were aligned to video start times."
    current = f["session_description"][()]
    current = current.decode() if isinstance(current, (bytes, bytearray)) else str(current)
    if tag not in current:
        f["session_description"][()] = (current + tag).encode("utf-8")

# --- verify after write ---
io = NWBHDF5IO(path, "r")
nwb = io.read()
epochs_df_after = nwb.intervals["epochs"].to_dataframe()
print("\nEpoch times (after):\n", epochs_df_after)
print("\nEpoch ranges from->to:")
for (o_start, o_stop), (n_start, n_stop) in zip(old_ranges, new_ranges):
    print(f"{o_start:.3f}-{o_stop:.3f} -> {n_start:.3f}-{n_stop:.3f}")
print("\nSession description:\n", nwb.session_description)
io.close()


In [ ]:
# mapping the video stamps to align with epoch time stamps ------------------- still in progress


from pynwb import NWBHDF5IO, TimeSeries
import numpy as np
import pandas as pd

# -------------------------------
# PARAMETERS
# -------------------------------
nwb_file_path = "/home/supraja/Documents/spyglass/NA18/raw/na1820250815.nwb"

# -------------------------------
# HELPER FUNCTIONS
# -------------------------------
def confirm(prompt):
    """Ask user for yes/no input."""
    x = input(prompt + " (y/n): ").strip().lower()
    return x == 'y'

def check_dropped_frames(t_array, tolerance=0.001):
    """
    Check for dropped frames based on differences between consecutive timestamps.
    tolerance: any interval larger than median + tolerance is counted as dropped frame
    """
    dt = np.diff(t_array)
    median_dt = np.median(dt)
    n_dropped = np.sum(dt > median_dt + tolerance)
    return n_dropped

# -------------------------------
# LOAD NWB
# -------------------------------
io = NWBHDF5IO(nwb_file_path, 'r+')
nwb = io.read()
video_module = nwb.processing['video_files']['video'].time_series

# -------------------------------
# EXTRACT VIDEO TIMESTAMPS
# -------------------------------
video_rows = []
for key in list(video_module.keys()):
    ts = video_module[key]
    t_array = np.array(ts.get_timestamps())
    video_rows.append({
        "video": key,
        "start_time": float(t_array[0]),
        "end_time": float(t_array[-1]),
        "frames": len(t_array)
    })

video_df = pd.DataFrame(video_rows)
print("=== Video Timestamp Table ===")
print(video_df)

# -------------------------------
# SHOW EPOCH TABLE
# -------------------------------
epochs_df = nwb.intervals["epochs"].to_dataframe()
print("\n=== Epoch Table ===")
print(epochs_df)

# -------------------------------
# PRINT COMPARISON FOR USER INSPECTION
# -------------------------------
print("\nVideo vs Epoch start/end times:")
for key in list(video_module.keys()):
    ts = video_module[key]
    t_array = np.array(ts.get_timestamps())
    video_start, video_end = t_array[0], t_array[-1]
    epoch_start, epoch_end = epochs_df.iloc[0]['start_time'], epochs_df.iloc[0]['stop_time']
    print(f"Video '{key}': start={video_start:.3f}s, end={video_end:.3f}s")
    print(f"Epoch 0: start={epoch_start:.3f}s, end={epoch_end:.3f}s")

# -------------------------------
# CONFIRM MAPPING
# -------------------------------
if confirm("\nDo you want to map timestamps for these videos?"):
    print("\nMapping timestamps...\n")
    mapped_videos = []
    video_keys = list(video_module.keys())  # snapshot to avoid mutating during iteration

    for key in video_keys:
        ts = video_module[key]
        t_array = np.array(ts.get_timestamps())

        # Compute offset to align video start to epoch start
        epoch_start = epochs_df.iloc[0]['start_time']
        video_start = t_array[0]
        offset = epoch_start - video_start
        t_mapped = t_array + offset

        # Check end frame alignment
        video_end = t_mapped[-1]
        epoch_end = epochs_df.iloc[0]['stop_time']
        end_diff = abs(video_end - epoch_end)
        if end_diff > 1e-3:
            print(f"[WARNING] Video '{key}' end time {video_end:.3f}s does not match epoch end {epoch_end:.3f}s (diff {end_diff:.3f}s)")

        # Create new TimeSeries with mapped timestamps
        mapped_ts = TimeSeries(
            name=ts.name + "_mapped",
            data=ts.data[:],
            unit=getattr(ts, 'unit', 'NA'),
            timestamps=t_mapped,
            description=(ts.description or "") + " [Mapped to epoch start]"
        )

        # Add mapped TimeSeries to NWB processing module
        nwb.processing['video_files']['video'].add_timeseries(mapped_ts)
        mapped_videos.append(key)
        print(f"✓ Mapped timestamps for video '{key}'")

    # -------------------------------
    # CHECK DROPPED FRAMES AFTER MAPPING
    # -------------------------------
    print("\nChecking for dropped frames after mapping:")
    for key in mapped_videos:
        ts_mapped = nwb.processing['video_files']['video'][key + "_mapped"]
        t_array = np.array(ts_mapped.timestamps)
        n_dropped = check_dropped_frames(t_array)
        print(f"Video '{key}_mapped': dropped frames = {n_dropped}, total frames = {len(t_array)}")

    # -------------------------------
    # ADD COMMENT TO SESSION DESCRIPTION
    # -------------------------------
    comment = "\n[INFO] Video timestamps were aligned to epoch start times."
    desc = nwb.session_description or ""
    if comment not in desc:
        desc = desc + comment
    nwb.fields["session_description"] = desc
    io.write(nwb)
    try:
        import h5py
        with h5py.File(nwb_file_path, "r+") as f:
            current = f["session_description"][()].decode("utf-8")
            base = current.split("\n[INFO]")[0].rstrip()
            target = base + comment
            if current != target:
                f["session_description"][()] = target.encode("utf-8")
    except Exception as e:
        print(f"[WARN] Could not persist session_description via h5py: {e}")
    print("\n✓ Added mapping comment to nwb.session_description")
else:
    print("No mapping applied. User canceled.")

io.close()



In [ ]:
# for mapping the video time series to epoch
import h5py
import numpy as np

path = "/home/supraja/Documents/spyglass/NA25/raw/na2520250919.nwb"

# video_epoch_mapped is a dict {video_name: mapped_timestamps_array}
with h5py.File(path, 'a') as f:
    video_group = f['processing']['video_files']['video']

    for video_name, new_timestamps in video_epoch_mapped.items():
        ts_group = video_group[video_name]  # each video TimeSeries is a subgroup
        print(f"{video_name} current timestamps (first 5):", ts_group['timestamps'][:5])

        # Overwrite timestamps
        ts_group['timestamps'][:] = new_timestamps

        print(f"{video_name} new timestamps (first 5):", ts_group['timestamps'][:5])


In [ ]:
# for mapping the epoch time series to video

import h5py
import numpy as np
from pynwb import NWBHDF5IO

path = "/home/sambray/Downloads/minirec20230622_edits.nwb" # path to your NWB file

# list of the start and stop time vaules you want in the file
new_start_times = np.array([12345, 23456])
new_stop_times = np.array([12350, 23460])

# open the file and overwrite the start and stop times
with h5py.File(path, 'a') as f:
    print("current list of start times:", f['intervals']['epochs']['start_time'][:])
    print("current list of stop times:", f['intervals']['epochs']['stop_time'][:])
    f['intervals']['epochs']['start_time'][:] = new_start_times
    f['intervals']['epochs']['stop_time'][:] = new_stop_times
    print("new list of start times:", f['intervals']['epochs']['start_time'][:])
    print("new list of stop times:", f['intervals']['epochs']['stop_time'][:])

# verify the changes using pynwb
io = NWBHDF5IO(path, 'r')
nwbfile = io.read()
nwbfile.intervals['epochs'].to_dataframe()